In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
import os
from datetime import datetime
    
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from src.dataset import SatalliteDataset, get_train_transforms, get_eval_transforms

def save_model(model: torch.nn.Module):
    model_timestamp = datetime.now().strftime("%Y-%m-%d-%H:%M")
    model_name = f"{model.__class__.__name__}_{model_timestamp}.pth"
    torch.save(model.state_dict(), os.path.join("models", model_name))

save_model(model)

## DataLoaders

In [47]:
BATCH_SIZE = 16

train_transforms = get_train_transforms()
eval_transforms = get_eval_transforms()

train_loader = DataLoader(
    dataset=SatalliteDataset(file="artifacts/train.csv", transforms=train_transforms),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    dataset=SatalliteDataset(file="artifacts/valid.csv", transforms=eval_transforms),
    batch_size=BATCH_SIZE,
)

test_loader = DataLoader(
    dataset=SatalliteDataset(file="artifacts/test.csv", transforms=eval_transforms),
    batch_size=BATCH_SIZE,
)

## Training

In [46]:
from torch.optim import Adam

from src.experiment import Experiment
from src.models.resnet import ResNet

model = ResNet(in_channels=3, n_classes=4)

exp = Experiment(
    model=model,
    device=torch.device("cuda"),
    optimizer=Adam(model.parameters()),
    criterion=torch.nn.CrossEntropyLoss(),
)

results = exp.train(
    train_loader=train_loader,
    valid_loader=valid_loader,
    epochs=40,
    max_lr=0.01,
    grad_clip=0.1,
)

Epoch 01/40 | train_loss=0.4341 | train_acc=0.8288 | valid_loss=0.4122 | valid_acc=0.8224
Epoch 02/40 | train_loss=0.3343 | train_acc=0.8741 | valid_loss=0.1824 | valid_acc=0.9201
Epoch 03/40 | train_loss=0.3857 | train_acc=0.8668 | valid_loss=0.5011 | valid_acc=0.8064
Epoch 04/40 | train_loss=0.3769 | train_acc=0.8699 | valid_loss=0.6312 | valid_acc=0.7886
Epoch 05/40 | train_loss=0.4157 | train_acc=0.8619 | valid_loss=0.6040 | valid_acc=0.7318
Epoch 06/40 | train_loss=0.3583 | train_acc=0.8761 | valid_loss=1.3158 | valid_acc=0.6483
Epoch 07/40 | train_loss=0.3362 | train_acc=0.8870 | valid_loss=1.0199 | valid_acc=0.8117
Epoch 08/40 | train_loss=0.3410 | train_acc=0.8859 | valid_loss=1.5740 | valid_acc=0.7762
Epoch 09/40 | train_loss=0.2818 | train_acc=0.8974 | valid_loss=0.2555 | valid_acc=0.9023
Epoch 10/40 | train_loss=0.2481 | train_acc=0.9161 | valid_loss=1.2262 | valid_acc=0.8082
Epoch 11/40 | train_loss=0.2739 | train_acc=0.9056 | valid_loss=0.1110 | valid_acc=0.9734
Epoch 12/4

In [48]:
exp.evaluate(loader=test_loader)

(0.01088314591362042, 0.99822695035461)

In [57]:
save_model(model)